
# Kokoro phoneme navigator demo

This notebook is a prototype for a pronunciation-editing interface that treats a Kokoro phoneme string as a local navigation problem rather than as raw text editing.

The notebook requires a running Kokoro-FastAPI server.
There is no mock mode.
Failures from Kokoro are shown directly in the interface.

## Desired behaviour

1. Enter a word or short phrase at the top of the interface.
2. Press **Phonemise**.
3. The notebook calls Kokoro-FastAPI `/dev/phonemize`.
4. The returned phoneme string is tokenised into sensible editable units.
   Common English diphthongs, affricates, and stress marks are grouped or separated explicitly.
5. The main pane displays the tokenised phoneme string, for example:

   ```text
   rough

   ɹ   ˈ   [ʌ]   f
   ```

6. Select a token by mouse click or with the left and right arrow keys.
7. The candidate pane shows local phonetic moves around the selected token:

   ```text
             [ə] weaker / reduced
   [ɒ] rounder     [ʌ]     [ɑ] darker / backer
             [ɐ] more open
   ```

8. Apply moves by mouse click or keyboard:
   * `↑` applies the upward candidate.
   * `↓` applies the downward candidate.
   * `a` and `d` apply left and right candidates.
   * `q`, `e`, `c`, and `x` apply diagonal candidates if present.
   * `←` and `→` move between phoneme tokens.
   * `space` speaks the current phoneme string.
   * `z` undoes the last edit.
9. After an edit, the notebook calls Kokoro-FastAPI `/dev/generate_from_phonemes` and plays the updated audio if **speak after edit** is enabled.

## Backend assumptions

The upstream Kokoro-FastAPI README documents the relevant development endpoints:

```python
requests.post(
    "http://localhost:8880/dev/phonemize",
    json={"text": text, "language": language}
)
```

and

```python
requests.post(
    "http://localhost:8880/dev/generate_from_phonemes",
    json={"phonemes": phonemes, "voice": voice},
    headers={"Accept": "audio/wav"}
)
```

The README states that `"a"` is the language code for American English.
The default notebook settings are therefore:

```text
Kokoro URL: http://localhost:8880
Language:   a
Voice:      af_bella
```

## Minimal Python dependencies

Run this in the notebook environment if needed:

```bash
python -m pip install ipywidgets ipyevents requests
```

For JupyterLab, make sure widgets are enabled in the environment from which the notebook kernel runs.


In [37]:

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable, Literal

import numpy as np
import requests
import ipywidgets as widgets
from IPython.display import Audio, HTML, display

try:
    from ipyevents import Event
except ImportError as exc:
    raise ImportError(
        "This notebook requires ipyevents for arrow-key navigation. "
        "Install with: python -m pip install ipyevents"
    ) from exc


In [38]:

Direction = Literal[
    "up",
    "down",
    "left",
    "right",
    "up_left",
    "up_right",
    "down_left",
    "down_right",
]
TokenKind = Literal["vowel", "consonant", "stress", "diacritic", "separator", "unknown"]
MoveAction = Literal[
    "replace",
    "delete",
    "move_stress_left",
    "move_stress_right",
    "insert_stress_before",
    "insert_copy_before",
]


@dataclass(frozen=True)
class PhonemeToken:
    """A display and editing unit in a phoneme string.

    Attributes:
        text: The IPA fragment represented by the token.
        kind: Coarse phonological class used to select candidate moves.
        editable: Whether the token can be selected for editing.
    """

    text: str
    kind: TokenKind
    editable: bool = True


@dataclass(frozen=True)
class CandidateMove:
    """A local edit that can be applied to a selected phoneme token.

    Attributes:
        direction: The spatial direction in the local navigation lattice.
        replacement: Replacement IPA text for replacement edits.
        label: Human-readable perceptual or articulatory description.
        action: Edit operation applied when the move is selected.
    """

    direction: Direction
    replacement: str
    label: str
    action: MoveAction = "replace"


@dataclass(frozen=True)
class DerivedCandidate:
    """A possibly multi-step candidate shown in the expanded lattice.

    Attributes:
        replacement: IPA text reached at the lattice coordinate.
        path: Ordered replacement moves used to reach it.
    """

    replacement: str
    path: tuple[CandidateMove, ...]


VOWELS: set[str] = {
    "i",
    "ɪ",
    "e",
    "eɪ",
    "ɛ",
    "æ",
    "a",
    "ɑ",
    "ɒ",
    "ɔ",
    "o",
    "oʊ",
    "ʊ",
    "u",
    "ə",
    "ɚ",
    "ɜ",
    "ɝ",
    "ʌ",
    "ɐ",
    "aɪ",
    "aʊ",
    "ɔɪ",
    "ɪə",
    "eə",
    "ʊə",
}

STRESS_MARKS: set[str] = {"ˈ", "ˌ"}
DIACRITICS: set[str] = {"ː", "ˑ", "̃", "̩", "ʰ", "ʲ", "ʷ"}
SEPARATORS: set[str] = {" ", "\t", "\n", ".", ",", "·"}

MULTI_CHAR_TOKENS: list[str] = sorted(
    [
        "tʃ",
        "dʒ",
        "aɪ",
        "aʊ",
        "ɔɪ",
        "eɪ",
        "oʊ",
        "ɪə",
        "eə",
        "ʊə",
        "ɝ",
        "ɚ",
    ],
    key=len,
    reverse=True,
)


def classify_ipa_token(text: str) -> TokenKind:
    """Classify a token into a coarse phonological class.

    Args:
        text: The IPA token.

    Returns:
        The token kind used by the navigator.
    """

    if text in VOWELS:
        return "vowel"
    if text in STRESS_MARKS:
        return "stress"
    if text in DIACRITICS:
        return "diacritic"
    if text in SEPARATORS:
        return "separator"
    if not text.strip():
        return "separator"
    return "consonant"


def tokenise_ipa(phonemes: str) -> list[PhonemeToken]:
    """Tokenise an IPA-like Kokoro phoneme string into editable units.

    This is deliberately conservative. It groups common English diphthongs and
    affricates, keeps stress marks as separate editable units, and otherwise
    falls back to one Unicode code point per token.

    Args:
        phonemes: The phoneme string returned by Kokoro.

    Returns:
        A list of phoneme tokens.
    """

    tokens: list[PhonemeToken] = []
    index: int = 0

    while index < len(phonemes):
        matched: str | None = None
        for candidate in MULTI_CHAR_TOKENS:
            if phonemes.startswith(candidate, index):
                matched = candidate
                break

        if matched is None:
            matched = phonemes[index]

        kind: TokenKind = classify_ipa_token(matched)
        editable: bool = kind != "separator"
        tokens.append(PhonemeToken(text=matched, kind=kind, editable=editable))
        index += len(matched)

    return tokens


def detokenise_ipa(tokens: list[PhonemeToken]) -> str:
    """Convert phoneme tokens back to a Kokoro phoneme string.

    Args:
        tokens: The phoneme tokens in display order.

    Returns:
        The concatenated phoneme string.
    """

    return "".join(token.text for token in tokens)


@dataclass(frozen=True)
class LocalProjectedNeighbourhood:
    """A local 2D vowel neighbourhood projected from feature space."""

    center: str
    positions: dict[str, tuple[float, float]]
    axis_labels: dict[str, str]
    hover_labels: dict[str, str]


FEATURE_NAMES: tuple[str, ...] = (
    "openness",
    "backness",
    "rounding",
    "rhoticity",
    "diphthongality",
)

FEATURE_LABELS: dict[str, tuple[str, str]] = {
    "openness": ("closer", "more open"),
    "backness": ("fronter", "backer / darker"),
    "rounding": ("less rounded", "rounder"),
    "rhoticity": ("less rhotic", "more rhotic"),
    "diphthongality": ("steadier", "more diphthongal"),
}

MONOPHTHONG_FEATURES: dict[str, tuple[float, float, float, float, float]] = {
    "i": (0.05, 0.05, 0.0, 0.0, 0.0),
    "ɪ": (0.20, 0.15, 0.0, 0.0, 0.0),
    "e": (0.28, 0.15, 0.0, 0.0, 0.0),
    "ɛ": (0.45, 0.22, 0.0, 0.0, 0.0),
    "æ": (0.80, 0.15, 0.0, 0.0, 0.0),
    "a": (0.98, 0.35, 0.0, 0.0, 0.0),
    "ɐ": (0.82, 0.50, 0.0, 0.0, 0.0),
    "ʌ": (0.65, 0.62, 0.0, 0.0, 0.0),
    "ə": (0.42, 0.50, 0.0, 0.0, 0.0),
    "ɜ": (0.48, 0.45, 0.0, 0.0, 0.0),
    "ɝ": (0.48, 0.45, 0.0, 1.0, 0.0),
    "ɚ": (0.42, 0.50, 0.0, 1.0, 0.0),
    "ɑ": (0.90, 0.92, 0.0, 0.0, 0.0),
    "ɒ": (0.88, 0.88, 1.0, 0.0, 0.0),
    "ɔ": (0.58, 0.88, 1.0, 0.0, 0.0),
    "o": (0.35, 0.82, 1.0, 0.0, 0.0),
    "ʊ": (0.26, 0.80, 1.0, 0.0, 0.0),
    "u": (0.08, 0.88, 1.0, 0.0, 0.0),
    "ɤ": (0.34, 0.72, 0.0, 0.0, 0.0),
}

COMPOSITE_VOWELS: dict[str, tuple[str, ...]] = {
    "eɪ": ("e", "ɪ"),
    "oʊ": ("o", "ʊ"),
    "aɪ": ("a", "ɪ"),
    "aʊ": ("a", "ʊ"),
    "ɔɪ": ("ɔ", "ɪ"),
    "ɪə": ("ɪ", "ə"),
    "eə": ("e", "ə"),
    "ʊə": ("ʊ", "ə"),
}

VOWEL_EXAMPLES: dict[str, tuple[str, str, str]] = {
    "i": ("EE", "see", "machine"),
    "ɪ": ("I", "bit", "sit"),
    "e": ("E", "bed", "many"),
    "ɛ": ("E", "dress", "best"),
    "æ": ("A", "apple", "cat"),
    "a": ("A", "father", "spa"),
    "ɐ": ("UH", "strut", "comma"),
    "ʌ": ("U", "cup", "luck"),
    "ə": ("A", "about", "sofa"),
    "ɜ": ("UR", "nurse", "word"),
    "ɝ": ("ER", "bird", "learn"),
    "ɚ": ("ER", "butter", "doctor"),
    "ɑ": ("AH", "father", "palm"),
    "ɒ": ("O", "lot", "off"),
    "ɔ": ("AW", "thought", "law"),
    "o": ("O", "go", "role"),
    "ʊ": ("OO", "book", "good"),
    "u": ("OO", "goose", "blue"),
    "ɤ": ("UH", "comma", "sofa"),
    "eɪ": ("A", "day", "face"),
    "oʊ": ("O", "go", "home"),
    "aɪ": ("I", "my", "time"),
    "aʊ": ("OW", "now", "mouth"),
    "ɔɪ": ("OY", "boy", "choice"),
    "ɪə": ("EAR", "near", "serious"),
    "eə": ("AIR", "square", "care"),
    "ʊə": ("OOR", "tour", "cure"),
}


def vowel_feature_vector(phoneme: str) -> np.ndarray | None:
    """Return a vowel feature vector when one is available."""

    if phoneme in MONOPHTHONG_FEATURES:
        return np.array(MONOPHTHONG_FEATURES[phoneme], dtype=float)
    if phoneme in COMPOSITE_VOWELS:
        component_vectors: list[np.ndarray] = []
        for component in COMPOSITE_VOWELS[phoneme]:
            vector = vowel_feature_vector(component)
            if vector is None:
                return None
            component_vectors.append(vector)
        composite: np.ndarray = np.mean(np.vstack(component_vectors), axis=0)
        composite[4] = 1.0
        return composite
    return None


def describe_vowel_example(phoneme: str) -> str:
    """Return a short human-readable gloss for a vowel symbol."""

    example = VOWEL_EXAMPLES.get(phoneme)
    if example is None:
        return f"IPA vowel {phoneme}"
    spelling_hint, word_one, word_two = example
    return f"Like the {spelling_hint} in {word_one} or {word_two}."


def describe_vowel_change(source: str, target: str) -> str:
    """Return a short local description of a vowel change."""

    source_vector = vowel_feature_vector(source)
    target_vector = vowel_feature_vector(target)
    if source_vector is None or target_vector is None:
        return "nearby vowel"

    delta: np.ndarray = target_vector - source_vector
    dimensions: list[int] = list(np.argsort(np.abs(delta))[::-1])
    labels: list[str] = []
    for dimension in dimensions:
        magnitude: float = float(abs(delta[dimension]))
        if magnitude < 0.12:
            continue
        lower_label, upper_label = FEATURE_LABELS[FEATURE_NAMES[dimension]]
        labels.append(upper_label if delta[dimension] > 0 else lower_label)
        if len(labels) == 2:
            break

    return ", ".join(labels) if labels else "nearby vowel"


def infer_axis_labels(components: np.ndarray) -> dict[str, str]:
    """Infer rough local directional labels from PCA component loadings."""

    labels: dict[str, str] = {}
    axis_names: tuple[tuple[str, str, int], ...] = (
        ("left", "right", 0),
        ("down", "up", 1),
    )
    for negative_key, positive_key, component_index in axis_names:
        if component_index >= components.shape[1]:
            labels[negative_key] = "that way"
            labels[positive_key] = "the other way"
            continue
        dominant_dimension: int = int(np.argmax(np.abs(components[:, component_index])))
        lower_label, upper_label = FEATURE_LABELS[FEATURE_NAMES[dominant_dimension]]
        if components[dominant_dimension, component_index] >= 0:
            labels[negative_key] = lower_label
            labels[positive_key] = upper_label
        else:
            labels[negative_key] = upper_label
            labels[positive_key] = lower_label
    return labels


def build_projected_vowel_neighbourhood(
    phoneme: str,
    max_neighbours: int = 8,
) -> LocalProjectedNeighbourhood | None:
    """Project a local vowel neighbourhood into 2D with PCA."""

    center_vector = vowel_feature_vector(phoneme)
    if center_vector is None:
        return None

    candidates: list[tuple[float, str, np.ndarray]] = []
    for candidate in sorted(VOWELS):
        if candidate == phoneme:
            continue
        vector = vowel_feature_vector(candidate)
        if vector is None:
            continue
        distance: float = float(np.linalg.norm(vector - center_vector))
        candidates.append((distance, candidate, vector))

    candidates.sort(key=lambda item: item[0])
    selected_candidates = candidates[:max_neighbours]
    labels: list[str] = [phoneme] + [candidate for _, candidate, _ in selected_candidates]
    vectors: list[np.ndarray] = [center_vector] + [vector for _, _, vector in selected_candidates]

    matrix: np.ndarray = np.vstack(vectors)
    centred: np.ndarray = matrix - np.mean(matrix, axis=0)
    covariance: np.ndarray = np.cov(centred, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order: np.ndarray = np.argsort(eigenvalues)[::-1]
    components: np.ndarray = eigenvectors[:, order[:2]]
    if components.ndim == 1:
        components = components[:, np.newaxis]
    if components.shape[1] == 1:
        components = np.hstack([components, np.zeros((components.shape[0], 1), dtype=float)])

    projected: np.ndarray = centred @ components
    projected = projected - projected[0]
    max_abs: float = float(np.max(np.abs(projected)))
    if max_abs > 0:
        projected = projected / max_abs

    axis_labels: dict[str, str] = infer_axis_labels(components)
    positions: dict[str, tuple[float, float]] = {
        label: (float(projected[index, 0]), float(projected[index, 1]))
        for index, label in enumerate(labels)
    }
    hover_labels: dict[str, str] = {
        label: describe_vowel_change(phoneme, label)
        for label in labels
    }
    hover_labels[phoneme] = "selected vowel"
    return LocalProjectedNeighbourhood(
        center=phoneme,
        positions=positions,
        axis_labels=axis_labels,
        hover_labels=hover_labels,
    )


def choose_projected_vowel_move(phoneme: str, direction: Direction) -> CandidateMove | None:
    """Choose the nearest projected vowel move in a geometric direction."""

    if direction not in {"up", "down", "left", "right"}:
        return None

    neighbourhood = build_projected_vowel_neighbourhood(phoneme)
    if neighbourhood is None:
        return None

    direction_vectors: dict[Direction, np.ndarray] = {
        "up": np.array([0.0, 1.0]),
        "down": np.array([0.0, -1.0]),
        "left": np.array([-1.0, 0.0]),
        "right": np.array([1.0, 0.0]),
    }
    target_direction: np.ndarray = direction_vectors[direction]
    best_score: float | None = None
    best_candidate: str | None = None
    for candidate, position in neighbourhood.positions.items():
        if candidate == phoneme:
            continue
        vector = np.array(position, dtype=float)
        norm: float = float(np.linalg.norm(vector))
        if norm < 1e-6:
            continue
        alignment: float = float(np.dot(vector, target_direction) / norm)
        if alignment < 0.2:
            continue
        score: float = alignment - 0.18 * norm
        if best_score is None or score > best_score:
            best_score = score
            best_candidate = candidate

    if best_candidate is None:
        return None
    return CandidateMove(
        direction,
        best_candidate,
        neighbourhood.hover_labels[best_candidate],
    )


def projected_grid_positions(
    positions: dict[str, tuple[float, float]],
    grid_size: int = 7,
) -> dict[str, tuple[int, int]]:
    """Place projected nodes onto a small display grid."""

    placements: dict[str, tuple[int, int]] = {}
    occupied: set[tuple[int, int]] = set()
    ordered_nodes: list[str] = sorted(
        positions,
        key=lambda label: abs(positions[label][0]) + abs(positions[label][1]),
    )
    centre: int = grid_size // 2
    for label in ordered_nodes:
        x_value, y_value = positions[label]
        row = int(round((1.0 - y_value) * centre))
        col = int(round((x_value + 1.0) * centre))
        row = max(0, min(grid_size - 1, row))
        col = max(0, min(grid_size - 1, col))
        target_cell: tuple[int, int] | None = None
        for radius in range(grid_size):
            for delta_row in range(-radius, radius + 1):
                for delta_col in range(-radius, radius + 1):
                    candidate_cell = (row + delta_row, col + delta_col)
                    if not (0 <= candidate_cell[0] < grid_size and 0 <= candidate_cell[1] < grid_size):
                        continue
                    if candidate_cell in occupied:
                        continue
                    target_cell = candidate_cell
                    break
                if target_cell is not None:
                    break
            if target_cell is not None:
                break
        if target_cell is None:
            continue
        placements[label] = target_cell
        occupied.add(target_cell)

    return placements


In [39]:

CANDIDATES: dict[str, list[CandidateMove]] = {
    "ʌ": [
        CandidateMove("up", "ə", "weaker / reduced"),
        CandidateMove("down", "ɐ", "more open"),
        CandidateMove("left", "ɒ", "rounder"),
        CandidateMove("right", "ɑ", "darker / backer"),
    ],
    "ə": [
        CandidateMove("up", "ɪ", "closer"),
        CandidateMove("down", "ʌ", "stronger / fuller"),
        CandidateMove("left", "ɚ", "r-coloured"),
        CandidateMove("right", "ɐ", "more open"),
    ],
    "ɐ": [
        CandidateMove("up", "ʌ", "less open"),
        CandidateMove("left", "ɒ", "rounder"),
        CandidateMove("right", "ɑ", "backer"),
        CandidateMove("down", "a", "more open"),
    ],
    "ɑ": [
        CandidateMove("up", "ʌ", "centraler"),
        CandidateMove("left", "ɒ", "rounder"),
        CandidateMove("right", "æ", "fronter"),
        CandidateMove("down", "a", "more open"),
    ],
    "ɒ": [
        CandidateMove("up", "ɔ", "closer"),
        CandidateMove("right", "ɑ", "unrounded"),
        CandidateMove("down", "ɐ", "less rounded / central"),
    ],
    "æ": [
        CandidateMove("up", "ɛ", "closer"),
        CandidateMove("down", "a", "more open"),
        CandidateMove("left", "ɑ", "backer"),
        CandidateMove("right", "e", "tenser / fronter"),
    ],
    "ɛ": [
        CandidateMove("up", "ɪ", "closer"),
        CandidateMove("down", "æ", "more open"),
        CandidateMove("right", "e", "tenser"),
        CandidateMove("left", "ɜ", "centraler"),
    ],
    "ɪ": [
        CandidateMove("up", "i", "tenser / closer"),
        CandidateMove("down", "ɛ", "more open"),
        CandidateMove("left", "ɜ", "centraler"),
        CandidateMove("right", "e", "tenser"),
    ],
    "i": [
        CandidateMove("down", "ɪ", "laxer"),
        CandidateMove("left", "ɨ", "centraler"),
    ],
    "ʊ": [
        CandidateMove("up", "u", "tenser / closer"),
        CandidateMove("down", "ɔ", "more open"),
        CandidateMove("left", "ɪ", "less rounded"),
        CandidateMove("right", "oʊ", "tenser / diphthongal"),
    ],
    "u": [
        CandidateMove("down", "ʊ", "laxer"),
        CandidateMove("left", "i", "less rounded / fronter"),
    ],
    "ɔ": [
        CandidateMove("up", "oʊ", "closer / diphthongal"),
        CandidateMove("down", "ɒ", "more open"),
        CandidateMove("right", "ɑ", "unrounded"),
    ],
    "oʊ": [
        CandidateMove("down", "ɔ", "less diphthongal"),
        CandidateMove("left", "ʊ", "shorter / laxer"),
    ],
    "eɪ": [
        CandidateMove("down", "ɛ", "less diphthongal"),
        CandidateMove("left", "ɪ", "shorter / laxer"),
    ],
    "aɪ": [
        CandidateMove("up", "æ", "less diphthongal"),
        CandidateMove("left", "ɑ", "backer onset"),
        CandidateMove("right", "eɪ", "closer onset"),
    ],
    "aʊ": [
        CandidateMove("up", "ɑ", "less diphthongal"),
        CandidateMove("left", "æ", "fronter onset"),
        CandidateMove("right", "oʊ", "closer / rounder"),
    ],
    "ɔɪ": [
        CandidateMove("down", "ɔ", "less diphthongal"),
        CandidateMove("right", "oʊ", "rounder / smoother"),
    ],
    "ɹ": [
        CandidateMove("up", "ɾ", "tap-like"),
        CandidateMove("down", "w", "weaker / more rounded"),
        CandidateMove("left", "r", "trilled / harder"),
        CandidateMove("right", "ɻ", "retroflex"),
    ],
    "r": [
        CandidateMove("right", "ɹ", "English approximant"),
        CandidateMove("down", "ɾ", "tap-like"),
    ],
    "ɾ": [
        CandidateMove("up", "r", "trilled / harder"),
        CandidateMove("down", "ɹ", "approximant"),
    ],
    "f": [
        CandidateMove("up", "v", "voiced"),
        CandidateMove("left", "θ", "dental"),
        CandidateMove("right", "p", "stop-like / stronger"),
        CandidateMove("down", "h", "softer"),
    ],
    "v": [
        CandidateMove("down", "f", "devoiced"),
        CandidateMove("right", "b", "stop-like / stronger"),
    ],
    "s": [
        CandidateMove("up", "z", "voiced"),
        CandidateMove("left", "θ", "dental"),
        CandidateMove("right", "ʃ", "postalveolar"),
    ],
    "z": [
        CandidateMove("down", "s", "devoiced"),
        CandidateMove("right", "ʒ", "postalveolar"),
    ],
    "ʃ": [
        CandidateMove("left", "s", "fronted"),
        CandidateMove("up", "ʒ", "voiced"),
        CandidateMove("right", "tʃ", "affricated"),
    ],
    "ʒ": [
        CandidateMove("down", "ʃ", "devoiced"),
        CandidateMove("right", "dʒ", "affricated"),
    ],
    "tʃ": [
        CandidateMove("left", "ʃ", "less stop-like"),
        CandidateMove("up", "dʒ", "voiced"),
        CandidateMove("right", "t", "less fricative"),
    ],
    "dʒ": [
        CandidateMove("down", "tʃ", "devoiced"),
        CandidateMove("left", "ʒ", "less stop-like"),
        CandidateMove("right", "d", "less fricative"),
    ],
    "t": [
        CandidateMove("up", "d", "voiced"),
        CandidateMove("left", "θ", "dental fricative"),
        CandidateMove("right", "tʃ", "affricated"),
        CandidateMove("down", "ʔ", "glottalised"),
    ],
    "d": [
        CandidateMove("down", "t", "devoiced"),
        CandidateMove("right", "dʒ", "affricated"),
    ],
    "p": [
        CandidateMove("up", "b", "voiced"),
        CandidateMove("left", "f", "fricative"),
        CandidateMove("down", "pʰ", "aspirated"),
    ],
    "b": [
        CandidateMove("down", "p", "devoiced"),
        CandidateMove("left", "v", "fricative"),
    ],
    "k": [
        CandidateMove("up", "g", "voiced"),
        CandidateMove("left", "x", "fricative"),
        CandidateMove("down", "kʰ", "aspirated"),
    ],
    "g": [
        CandidateMove("down", "k", "devoiced"),
        CandidateMove("left", "ɣ", "fricative"),
    ],
    "m": [
        CandidateMove("right", "n", "alveolar nasal"),
        CandidateMove("up", "b", "oral stop"),
    ],
    "n": [
        CandidateMove("left", "m", "bilabial nasal"),
        CandidateMove("right", "ŋ", "velar nasal"),
        CandidateMove("up", "d", "oral stop"),
    ],
    "ŋ": [
        CandidateMove("left", "n", "alveolar nasal"),
        CandidateMove("up", "g", "oral stop"),
    ],
    "l": [
        CandidateMove("up", "ɫ", "darker"),
        CandidateMove("down", "w", "more rounded / weaker"),
        CandidateMove("right", "ɹ", "r-like"),
    ],
    "w": [
        CandidateMove("up", "ɹ", "r-like"),
        CandidateMove("left", "ʊ", "vocalic"),
        CandidateMove("right", "v", "fricative"),
    ],
    "j": [
        CandidateMove("down", "i", "vocalic"),
        CandidateMove("right", "ʒ", "fricative"),
    ],
    "ˈ": [
        CandidateMove("up", "ˌ", "weaker stress"),
        CandidateMove("down", "", "remove stress", action="delete"),
        CandidateMove("left", "", "move stress left", action="move_stress_left"),
        CandidateMove("right", "", "move stress right", action="move_stress_right"),
    ],
    "ˌ": [
        CandidateMove("up", "ˈ", "stronger stress"),
        CandidateMove("down", "", "remove stress", action="delete"),
        CandidateMove("left", "", "move stress left", action="move_stress_left"),
        CandidateMove("right", "", "move stress right", action="move_stress_right"),
    ],
}


def remove_one_aspiration_marker(text: str) -> str:
    """Return a slightly less aspirated variant when one is available."""

    if len(text) > 1 and text.endswith("ʰ"):
        return text[:-1]
    if len(text) > 1 and text.endswith("h"):
        return text[:-1]
    return text


def generic_candidates(token: PhonemeToken) -> list[CandidateMove]:
    """Return fallback candidates for tokens not yet in the explicit table.

    Args:
        token: The selected token.

    Returns:
        A list of candidate moves.
    """

    if token.kind == "vowel":
        return [
            CandidateMove("up", "ə", "reduce"),
            CandidateMove("down", "a", "open"),
            CandidateMove("left", "ɒ", "round / back"),
            CandidateMove("right", "æ", "front"),
        ]

    if token.kind == "stress":
        return CANDIDATES.get(token.text, [])

    if token.kind == "consonant":
        weaker: str = remove_one_aspiration_marker(token.text)
        weaker_label: str = "less aspirated" if weaker != token.text else "no generic weaker move"
        return [
            CandidateMove("up", token.text + "ʰ", "aspirated"),
            CandidateMove("down", weaker, weaker_label),
        ]

    return []


OPPOSITE_DIRECTIONS: dict[Direction, Direction] = {
    "up": "down",
    "down": "up",
    "left": "right",
    "right": "left",
    "up_left": "down_right",
    "up_right": "down_left",
    "down_left": "up_right",
    "down_right": "up_left",
}


def infer_reverse_replace_moves(token: PhonemeToken) -> list[CandidateMove]:
    """Infer reversible replacement moves missing from the explicit graph."""

    inferred: list[CandidateMove] = []
    for source_text, source_moves in CANDIDATES.items():
        for move in source_moves:
            if move.action != "replace":
                continue
            if move.replacement != token.text:
                continue

            opposite: Direction = OPPOSITE_DIRECTIONS[move.direction]
            inferred.append(
                CandidateMove(
                    opposite,
                    source_text,
                    f"reverse of {move.label}",
                )
            )

    return inferred


def get_candidate_moves(token: PhonemeToken) -> list[CandidateMove]:
    """Return candidate moves for the selected token.

    Args:
        token: The selected token.

    Returns:
        A list of local edits.
    """

    moves: list[CandidateMove] = list(CANDIDATES.get(token.text, generic_candidates(token)))
    seen_directions: set[Direction] = {move.direction for move in moves}
    for move in infer_reverse_replace_moves(token):
        if move.direction in seen_directions:
            continue
        moves.append(move)
        seen_directions.add(move.direction)

    return moves


def derive_candidate_lattice(
    token: PhonemeToken,
    radius: int = 2,
) -> dict[tuple[int, int], DerivedCandidate]:
    """Expand direct candidate moves into a small local lattice.

    Args:
        token: The selected token.
        radius: Maximum Manhattan step count per axis away from the centre.

    Returns:
        A mapping from lattice coordinates to derived candidates.
    """

    origin: tuple[int, int] = (0, 0)
    lattice: dict[tuple[int, int], DerivedCandidate] = {
        origin: DerivedCandidate(replacement=token.text, path=()),
    }
    queue: list[tuple[str, tuple[int, int], tuple[CandidateMove, ...]]] = [(token.text, origin, ())]

    while queue:
        current_text, current_coord, current_path = queue.pop(0)
        if len(current_path) >= radius:
            continue

        current_token: PhonemeToken = PhonemeToken(
            text=current_text,
            kind=classify_ipa_token(current_text),
            editable=True,
        )
        for move in get_candidate_moves(current_token):
            if move.action != "replace":
                continue

            delta_x, delta_y = DIRECTION_VECTORS[move.direction]
            next_coord: tuple[int, int] = (
                current_coord[0] + delta_x,
                current_coord[1] + delta_y,
            )
            if abs(next_coord[0]) > radius or abs(next_coord[1]) > radius:
                continue

            next_path: tuple[CandidateMove, ...] = current_path + (move,)
            existing: DerivedCandidate | None = lattice.get(next_coord)
            if existing is not None and len(existing.path) <= len(next_path):
                continue

            derived = DerivedCandidate(replacement=move.replacement, path=next_path)
            lattice[next_coord] = derived
            queue.append((move.replacement, next_coord, next_path))

    return lattice


In [40]:

class KokoroClient:
    """Small client for the Kokoro-FastAPI phoneme endpoints."""

    def __init__(self, base_url: str, timeout_s: float = 30.0) -> None:
        """Initialise the client.

        Args:
            base_url: Kokoro-FastAPI base URL, for example "http://localhost:8880".
            timeout_s: HTTP request timeout in seconds.
        """

        self.base_url: str = base_url.rstrip("/")
        self.timeout_s: float = timeout_s

    def phonemize(self, text: str, language: str = "a") -> tuple[str, list[Any]]:
        """Call `/dev/phonemize`.

        Args:
            text: Text to phonemise.
            language: Kokoro language code. The upstream README gives "a" for
                American English.

        Returns:
            The phoneme string and Kokoro token IDs.

        Raises:
            requests.HTTPError: If Kokoro returns a non-2xx response.
            KeyError: If the response lacks the expected fields.
        """

        response: requests.Response = requests.post(
            f"{self.base_url}/dev/phonemize",
            json={"text": text, "language": language},
            timeout=self.timeout_s,
        )
        response.raise_for_status()
        result: dict[str, Any] = response.json()
        return str(result["phonemes"]), list(result["tokens"])

    def generate_from_phonemes(self, phonemes: str, voice: str = "af_bella") -> bytes:
        """Call `/dev/generate_from_phonemes`.

        Args:
            phonemes: Kokoro phoneme string.
            voice: Kokoro voice name or weighted voice expression.

        Returns:
            WAV audio bytes.

        Raises:
            requests.HTTPError: If Kokoro returns a non-2xx response.
        """

        response: requests.Response = requests.post(
            f"{self.base_url}/dev/generate_from_phonemes",
            json={"phonemes": phonemes, "voice": voice},
            headers={"Accept": "audio/wav"},
            timeout=self.timeout_s,
        )
        response.raise_for_status()
        return bytes(response.content)



## Design notes

The candidate table is deliberately a first draft.
The central design assumption is that users usually know the direction of the auditory error better than they know the target IPA symbol.
Therefore each candidate carries both an IPA replacement and a perceptual label.

The table is not a universal IPA chart.
It is a small, task-specific graph of likely substitutions for Kokoro pronunciation repair.
That is the part of the prototype that needs empirical tuning.


In [41]:

# Optional smoke test for the tokeniser and candidate table.
# This cell does not call Kokoro.

examples: list[str] = [
    "ɹˈʌf",
    "mˈɪnət",
    "mˌɪnjˈut",
    "kˌɑntɹˈækt",
]
for example in examples:
    tokens: list[PhonemeToken] = tokenise_ipa(example)
    print(example, "→", [token.text for token in tokens])


ɹˈʌf → ['ɹ', 'ˈ', 'ʌ', 'f']
mˈɪnət → ['m', 'ˈ', 'ɪ', 'n', 'ə', 't']
mˌɪnjˈut → ['m', 'ˌ', 'ɪ', 'n', 'j', 'ˈ', 'u', 't']
kˌɑntɹˈækt → ['k', 'ˌ', 'ɑ', 'n', 't', 'ɹ', 'ˈ', 'æ', 'k', 't']


In [42]:

DIRECTION_LABELS: dict[Direction, str] = {
    "up": "↑",
    "down": "↓",
    "left": "←",
    "right": "→",
    "up_left": "↖",
    "up_right": "↗",
    "down_left": "↙",
    "down_right": "↘",
}

DIRECTION_VECTORS: dict[Direction, tuple[int, int]] = {
    "up": (0, -1),
    "down": (0, 1),
    "left": (-1, 0),
    "right": (1, 0),
    "up_left": (-1, -1),
    "up_right": (1, -1),
    "down_left": (-1, 1),
    "down_right": (1, 1),
}

LATTICE_RADIUS: int = 2


class PhonemeNavigatorApp:
    """Interactive notebook UI for local phoneme navigation."""

    def __init__(self) -> None:
        """Build the widgets and initialise empty state."""

        self.tokens: list[PhonemeToken] = []
        self.selected_index: int = 0
        self.kokoro_tokens: list[Any] = []
        self.history: list[list[PhonemeToken]] = []

        self.base_url_input: widgets.Text = widgets.Text(
            value="http://localhost:8880",
            description="Kokoro URL",
            layout=widgets.Layout(width="360px"),
        )
        self.voice_input: widgets.Text = widgets.Text(
            value="af_bella",
            description="Voice",
            layout=widgets.Layout(width="240px"),
        )
        self.language_input: widgets.Text = widgets.Text(
            value="a",
            description="Language",
            layout=widgets.Layout(width="160px"),
        )
        self.word_input: widgets.Text = widgets.Text(
            value="rough",
            description="Word",
            layout=widgets.Layout(width="320px"),
            continuous_update=False,
        )
        self.phonemize_button: widgets.Button = widgets.Button(
            description="Phonemise",
            button_style="primary",
            tooltip="Call Kokoro /dev/phonemize",
        )
        self.speak_button: widgets.Button = widgets.Button(
            description="Speak",
            tooltip="Call Kokoro /dev/generate_from_phonemes",
        )
        self.undo_button: widgets.Button = widgets.Button(
            description="Undo",
            tooltip="Undo the last edit",
        )
        self.autoplay_toggle: widgets.Checkbox = widgets.Checkbox(
            value=True,
            description="speak after edit",
            indent=False,
        )

        self.status_html: widgets.HTML = widgets.HTML(value="")
        self.current_phonemes_html: widgets.HTML = widgets.HTML(value="")
        self.token_row: widgets.HBox = widgets.HBox()
        self.candidate_pane: widgets.VBox = widgets.VBox()
        self.audio_out: widgets.Output = widgets.Output()
        self.debug_out: widgets.Output = widgets.Output(layout=widgets.Layout(max_height="180px", overflow="auto"))

        self.keyboard_target: widgets.HTML = widgets.HTML(
            value=(
                "<div tabindex='0' style='border: 1px solid #bbb; "
                "border-radius: 8px; padding: 8px; margin-top: 6px;'>"
                "<b>Keyboard focus pane</b>: click here, then use "
                "Q/E previous/next phoneme, W/A/S/D candidate moves, "
                "R add stress before, T delete, F add phoneme before, "
                "space to speak, z to undo."
                "</div>"
            )
        )

        controls: widgets.VBox = widgets.VBox(
            [
                widgets.HBox([self.base_url_input, self.voice_input, self.language_input]),
                widgets.HBox(
                    [
                        self.word_input,
                        self.phonemize_button,
                        self.speak_button,
                        self.undo_button,
                        self.autoplay_toggle,
                    ]
                ),
            ]
        )

        self.root: widgets.VBox = widgets.VBox(
            [
                controls,
                self.status_html,
                self.current_phonemes_html,
                self.token_row,
                self.candidate_pane,
                self.keyboard_target,
                self.audio_out,
                self.debug_out,
            ],
            layout=widgets.Layout(border="1px solid #ddd", padding="12px", width="100%"),
        )

        self.phonemize_button.on_click(self._on_phonemize_clicked)
        self.speak_button.on_click(self._on_speak_clicked)
        self.undo_button.on_click(self._on_undo_clicked)
        self.word_input.on_submit(self._on_word_submitted)

        self.key_events: Event = Event(
            source=self.keyboard_target,
            watched_events=["keydown"],
            prevent_default_action=True,
        )
        self.key_events.on_dom_event(self._on_key_event)

        self._set_status("Enter a word and press Phonemise.", kind="neutral")
        self._render()

    def display(self) -> None:
        """Display the app in the current notebook cell."""

        display(self.root)

    def client(self) -> KokoroClient:
        """Return a Kokoro client built from the current URL field.

        Returns:
            A Kokoro client.
        """

        return KokoroClient(base_url=self.base_url_input.value)

    def _set_status(self, message: str, kind: str = "neutral") -> None:
        """Set the status display.

        Args:
            message: Message to display.
            kind: Semantic status kind.
        """

        colour: str = {
            "neutral": "#555",
            "ok": "#145214",
            "warn": "#8a5a00",
            "error": "#8a1f11",
        }.get(kind, "#555")
        self.status_html.value = f"<div style='color: {colour}; margin: 6px 0;'>{message}</div>"

    def _on_word_submitted(self, change: widgets.Widget) -> None:
        """Handle Enter in the word input.

        Args:
            change: The widget event object.
        """

        self._phonemize_current_word()

    def _on_phonemize_clicked(self, button: widgets.Button) -> None:
        """Handle the Phonemise button.

        Args:
            button: The clicked button.
        """

        self._phonemize_current_word()

    def _on_speak_clicked(self, button: widgets.Button) -> None:
        """Handle the Speak button.

        Args:
            button: The clicked button.
        """

        self.speak_current_phonemes()

    def _on_undo_clicked(self, button: widgets.Button) -> None:
        """Handle the Undo button.

        Args:
            button: The clicked button.
        """

        self.undo()

    def _phonemize_current_word(self) -> None:
        """Phonemise the current word through Kokoro and refresh the UI."""

        text: str = self.word_input.value.strip()
        if not text:
            self._set_status("No word entered.", kind="warn")
            return

        self._set_status("Calling Kokoro /dev/phonemize ...", kind="neutral")
        try:
            phonemes, kokoro_tokens = self.client().phonemize(text=text, language=self.language_input.value.strip())
        except Exception as exc:
            self._set_status(f"Phonemize failed: {type(exc).__name__}: {exc}", kind="error")
            return

        self.tokens = tokenise_ipa(phonemes)
        self.kokoro_tokens = kokoro_tokens
        self.history = []
        self.selected_index = self._first_editable_index(default=0)
        self._set_status(f"Phonemized {text!r}.", kind="ok")
        self._render()
        self.speak_current_phonemes()

    def _first_editable_index(self, default: int = 0) -> int:
        """Return the first editable token index.

        Args:
            default: Fallback index when no editable token is present.

        Returns:
            The first editable index or the fallback.
        """

        for index, token in enumerate(self.tokens):
            if token.editable:
                return index
        return default

    def _selected_token(self) -> PhonemeToken | None:
        """Return the selected token if there is one.

        Returns:
            The selected token, or None.
        """

        if not self.tokens:
            return None
        if self.selected_index < 0 or self.selected_index >= len(self.tokens):
            return None
        return self.tokens[self.selected_index]

    def _render(self) -> None:
        """Render the phoneme row and candidate pane."""

        phonemes: str = detokenise_ipa(self.tokens)
        self.current_phonemes_html.value = (
            f"<div style='font-size: 18px; margin: 8px 0;'>"
            f"<b>Current phonemes:</b> <code>{phonemes}</code></div>"
        )

        token_buttons: list[widgets.Button] = []
        for index, token in enumerate(self.tokens):
            description: str = "␠" if token.text == " " else token.text
            button: widgets.Button = widgets.Button(
                description=description,
                tooltip=f"{index}: {token.kind}",
                disabled=not token.editable,
                layout=widgets.Layout(width="48px", height="42px", margin="2px"),
            )
            button.button_style = "info" if index == self.selected_index else ""
            button.on_click(self._make_token_click_handler(index))
            token_buttons.append(button)

        self.token_row.children = tuple(token_buttons)
        self._render_candidates()

    def _make_token_click_handler(self, index: int) -> Callable[[widgets.Button], None]:
        """Create a button callback for selecting a token.

        Args:
            index: Token index to select.

        Returns:
            A button callback.
        """

        def handler(button: widgets.Button) -> None:
            self.select_index(index)

        return handler

    def select_index(self, index: int) -> None:
        """Select a token index and rerender.

        Args:
            index: Index to select.
        """

        if not self.tokens:
            return

        index = max(0, min(index, len(self.tokens) - 1))
        if not self.tokens[index].editable:
            step: int = 1 if index >= self.selected_index else -1
            index = self._next_editable_index(start=index, step=step)

        self.selected_index = index
        self._render()

    def _next_editable_index(self, start: int, step: int) -> int:
        """Find the next editable token in a direction.

        Args:
            start: Starting index.
            step: Direction, either 1 or -1.

        Returns:
            The next editable index, or the current selected index if none exists.
        """

        if not self.tokens:
            return 0

        index: int = start
        while 0 <= index < len(self.tokens):
            if self.tokens[index].editable:
                return index
            index += step

        return self.selected_index

    def move_selection(self, step: int) -> None:
        """Move the selected token left or right.

        Args:
            step: Direction, either 1 or -1.
        """

        if not self.tokens:
            return
        target: int = self._next_editable_index(start=self.selected_index + step, step=step)
        self.selected_index = target
        self._render()

    def _render_candidates(self) -> None:
        """Render candidate moves around the selected token."""

        selected: PhonemeToken | None = self._selected_token()
        if selected is None:
            self.candidate_pane.children = (widgets.HTML("<i>No phoneme selected.</i>"),)
            return

        if selected.kind == "vowel":
            neighbourhood = build_projected_vowel_neighbourhood(selected.text)
            if neighbourhood is not None:
                self._render_projected_vowel_neighbourhood(selected, neighbourhood)
                return

        moves: list[CandidateMove] = get_candidate_moves(selected)
        move_by_direction: dict[Direction, CandidateMove] = {move.direction: move for move in moves}
        lattice: dict[tuple[int, int], DerivedCandidate] = derive_candidate_lattice(
            selected,
            radius=LATTICE_RADIUS,
        )

        def make_button(direction: Direction) -> widgets.Widget:
            move: CandidateMove | None = move_by_direction.get(direction)
            if move is None:
                return widgets.HTML(value="<div style='width: 124px; height: 58px;'></div>")

            replacement: str = "∅" if move.replacement == "" else move.replacement
            button: widgets.Button = widgets.Button(
                description=replacement,
                tooltip=(
                    f"{DIRECTION_LABELS[direction]} {direction.replace('_', ' ')} | "
                    f"{move.action}: {selected.text} → {replacement} | {move.label}"
                ),
                layout=widgets.Layout(width="124px", height="58px", margin="2px"),
            )
            button.on_click(self._make_candidate_click_handler(move))
            return button

        def make_lattice_button(x: int, y: int) -> widgets.Widget:
            if x == 0 and y == 0:
                return widgets.HTML(
                    value=(
                        "<div style='width: 112px; height: 58px; display: flex; "
                        "align-items: center; justify-content: center; font-size: 22px; "
                        "font-weight: bold; border: 1px solid #999; border-radius: 6px;'>"
                        f"[{selected.text}]"
                        "</div>"
                    )
                )

            derived: DerivedCandidate | None = lattice.get((x, y))
            if derived is None or not derived.path:
                return widgets.HTML(value="<div style='width: 112px; height: 58px;'></div>")

            path_text: str = " ".join(DIRECTION_LABELS[step.direction] for step in derived.path)
            tooltip: str = " | ".join(
                f"{DIRECTION_LABELS[step.direction]} {step.direction.replace('_', ' ')} -> {step.replacement}: {step.label}"
                for step in derived.path
            )
            button: widgets.Button = widgets.Button(
                description=derived.replacement,
                tooltip=f"path {path_text} | {tooltip}",
                layout=widgets.Layout(width="112px", height="58px", margin="2px"),
            )
            button.on_click(
                self._make_candidate_click_handler(
                    CandidateMove(
                        direction=derived.path[-1].direction,
                        replacement=derived.replacement,
                        label=tooltip,
                    )
                )
            )
            return button

        def make_action_button(
            key_label: str,
            title: str,
            tooltip: str,
            handler: Callable[[widgets.Button], None],
        ) -> widgets.Button:
            button: widgets.Button = widgets.Button(
                description=f"{key_label} {title}",
                tooltip=tooltip,
                layout=widgets.Layout(width="170px", height="42px", margin="2px"),
            )
            button.on_click(handler)
            return button

        top: widgets.HBox = widgets.HBox(
            [make_button("up_left"), make_button("up"), make_button("up_right")],
            layout=widgets.Layout(justify_content="center"),
        )
        middle: widgets.HBox = widgets.HBox(
            [make_button("left"), make_lattice_button(0, 0), make_button("right")],
            layout=widgets.Layout(justify_content="center"),
        )
        bottom: widgets.HBox = widgets.HBox(
            [make_button("down_left"), make_button("down"), make_button("down_right")],
            layout=widgets.Layout(justify_content="center"),
        )
        expanded_rows: list[widgets.HBox] = []
        for y in range(-LATTICE_RADIUS, LATTICE_RADIUS + 1):
            row: widgets.HBox = widgets.HBox(
                [make_lattice_button(x, y) for x in range(-LATTICE_RADIUS, LATTICE_RADIUS + 1)],
                layout=widgets.Layout(justify_content="center"),
            )
            expanded_rows.append(row)

        direction_legend: widgets.HTML = widgets.HTML(
            value=(
                "<div style='margin-top: 8px; line-height: 1.5;'>"
                "<b>Direction legend</b>: "
                "W = ↑ up, S = ↓ down, A = ← left, D = → right, "
                "Q = previous phoneme, E = next phoneme. "
                "Arrow meaning stays separate from the phoneme symbols shown in the grid."
                "</div>"
            )
        )
        lattice_caption: widgets.HTML = widgets.HTML(
            value=(
                "<div style='margin-top: 6px; color: #444;'>"
                "Each cell shows only the phoneme candidate. Direction information lives in the legend and button tooltip."
                "</div>"
            )
        )

        action_row: widgets.HBox = widgets.HBox(
            [
                make_action_button("Q", "Prev", "Select the previous editable phoneme.", lambda _button: self.move_selection(-1)),
                make_action_button("E", "Next", "Select the next editable phoneme.", lambda _button: self.move_selection(1)),
                make_action_button("R", "Stress Before", "Insert a primary stress mark before the selected phoneme.", self._on_insert_stress_clicked),
                make_action_button("T", "Delete", "Delete the selected phoneme.", self._on_delete_selected_clicked),
                make_action_button("F", "Add Before", "Insert a copy of the selected phoneme before itself so you can reshape it with W/A/S/D.", self._on_insert_copy_before_clicked),
            ],
            layout=widgets.Layout(justify_content="center", flex_flow="row wrap"),
        )

        self.candidate_pane.children = (
            widgets.HTML(value=f"<div style='margin-top: 12px;'><b>Candidate moves for</b> <code>{selected.text}</code></div>"),
            direction_legend,
            widgets.HTML(value="<div style='margin-top: 4px;'>Direct neighbours</div>"),
            top,
            middle,
            bottom,
            widgets.HTML(value="<div style='margin-top: 10px;'><b>Expanded phoneme lattice</b> (two steps out, including off-axis combinations)</div>"),
            lattice_caption,
            *expanded_rows,
            widgets.HTML(value="<div style='margin-top: 10px;'><b>Edit actions</b></div>"),
            action_row,
        )

    def _render_projected_vowel_neighbourhood(
        self,
        selected: PhonemeToken,
        neighbourhood: LocalProjectedNeighbourhood,
    ) -> None:
        """Render the vowel neighbourhood as a projected local map."""

        placements: dict[str, tuple[int, int]] = projected_grid_positions(neighbourhood.positions)
        reverse_lookup: dict[tuple[int, int], str] = {
            cell: label for label, cell in placements.items()
        }
        grid_size: int = 7
        cells: list[widgets.Widget] = []
        for row in range(grid_size):
            for col in range(grid_size):
                label: str | None = reverse_lookup.get((row, col))
                if label is None:
                    cells.append(widgets.HTML(value="<div style='width: 82px; height: 54px;'></div>"))
                    continue
                if label == selected.text:
                    cells.append(
                        widgets.HTML(
                            value=(
                                "<div style='width: 82px; height: 54px; display: flex; "
                                "align-items: center; justify-content: center; font-size: 24px; "
                                "font-weight: bold; border: 1px solid #999; border-radius: 6px;'>"
                                f"[{label}]"
                                "</div>"
                            )
                        )
                    )
                    continue
                tooltip: str = neighbourhood.hover_labels[label]
                button = widgets.Button(
                    description=label,
                    tooltip=tooltip,
                    layout=widgets.Layout(width="82px", height="54px", margin="2px"),
                )
                button.on_click(
                    self._make_candidate_click_handler(
                        CandidateMove("up", label, tooltip)
                    )
                )
                cells.append(button)

        grid: widgets.GridBox = widgets.GridBox(
            cells,
            layout=widgets.Layout(
                grid_template_columns="repeat(7, 82px)",
                grid_template_rows="repeat(7, 54px)",
                justify_content="center",
                align_items="center",
            ),
        )
        top_label: widgets.HTML = widgets.HTML(
            value=(
                f"<div style='text-align: center; margin-bottom: 6px; color: #444;'>"
                f"{neighbourhood.axis_labels['up']}"
                "</div>"
            )
        )
        bottom_label: widgets.HTML = widgets.HTML(
            value=(
                f"<div style='text-align: center; margin-top: 6px; color: #444;'>"
                f"{neighbourhood.axis_labels['down']}"
                "</div>"
            )
        )
        left_label: widgets.HTML = widgets.HTML(
            value=(
                f"<div style='width: 150px; text-align: right; padding-right: 10px; color: #444;'>"
                f"{neighbourhood.axis_labels['left']}"
                "</div>"
            )
        )
        right_label: widgets.HTML = widgets.HTML(
            value=(
                f"<div style='width: 150px; text-align: left; padding-left: 10px; color: #444;'>"
                f"{neighbourhood.axis_labels['right']}"
                "</div>"
            )
        )
        direction_legend: widgets.HTML = widgets.HTML(
            value=(
                "<div style='margin-top: 8px; line-height: 1.5;'>"
                "<b>Direction legend</b>: W/A/S/D moves through the projected neighbourhood, "
                "Q/E select previous or next phoneme, click any visible vowel to replace directly."
                "</div>"
            )
        )
        caption: widgets.HTML = widgets.HTML(
            value=(
                "<div style='margin-top: 6px; color: #444;'>"
                "This panel is built from local vowel feature neighbours and a 2D PCA projection. "
                "Use it as a rough navigational map, not a global IPA atlas."
                "</div>"
            )
        )
        action_row: widgets.HBox = widgets.HBox(
            [
                widgets.Button(description="Q Prev", layout=widgets.Layout(width="170px", height="42px", margin="2px")),
                widgets.Button(description="E Next", layout=widgets.Layout(width="170px", height="42px", margin="2px")),
                widgets.Button(description="R Stress Before", layout=widgets.Layout(width="170px", height="42px", margin="2px")),
                widgets.Button(description="T Delete", layout=widgets.Layout(width="170px", height="42px", margin="2px")),
                widgets.Button(description="F Add Before", layout=widgets.Layout(width="170px", height="42px", margin="2px")),
            ],
            layout=widgets.Layout(justify_content="center", flex_flow="row wrap"),
        )
        action_row.children[0].on_click(lambda _button: self.move_selection(-1))
        action_row.children[1].on_click(lambda _button: self.move_selection(1))
        action_row.children[2].on_click(self._on_insert_stress_clicked)
        action_row.children[3].on_click(self._on_delete_selected_clicked)
        action_row.children[4].on_click(self._on_insert_copy_before_clicked)
        middle_row: widgets.HBox = widgets.HBox(
            [left_label, grid, right_label],
            layout=widgets.Layout(justify_content="center", align_items="center"),
        )
        phoneme_description: widgets.HTML = widgets.HTML(
            value=(
                "<div style='margin-top: 4px; color: #444;'>"
                f"{describe_vowel_example(selected.text)}"
                "</div>"
            )
        )
        self.candidate_pane.children = (
            widgets.HTML(value=f"<div style='margin-top: 12px;'><b>Projected vowel neighbourhood for</b> <code>{selected.text}</code></div>"),
            phoneme_description,
            direction_legend,
            top_label,
            middle_row,
            bottom_label,
            caption,
            widgets.HTML(value="<div style='margin-top: 10px;'><b>Edit actions</b></div>"),
            action_row,
        )

    def _make_candidate_click_handler(self, move: CandidateMove) -> Callable[[widgets.Button], None]:
        """Create a candidate move callback.

        Args:
            move: Move to apply.

        Returns:
            A button callback.
        """

        def handler(button: widgets.Button) -> None:
            self.apply_move(move)

        return handler

    def apply_move_by_direction(self, direction: Direction) -> None:
        """Apply the move in a given direction if it exists.

        Args:
            direction: Direction to apply.
        """

        selected: PhonemeToken | None = self._selected_token()
        if selected is None:
            return

        if selected.kind == "vowel":
            projected_move = choose_projected_vowel_move(selected.text, direction)
            if projected_move is not None:
                self.apply_move(projected_move)
                return

        for move in get_candidate_moves(selected):
            if move.direction == direction:
                self.apply_move(move)
                return

        self._set_status(f"No {direction.replace('_', ' ')} move for {selected.text!r}.", kind="warn")

    def apply_move(self, move: CandidateMove) -> None:
        """Apply a candidate move to the selected token.

        Args:
            move: Move to apply.
        """

        if not self.tokens:
            return

        self.history.append(list(self.tokens))

        if move.action == "replace":
            old: PhonemeToken = self.tokens[self.selected_index]
            self.tokens[self.selected_index] = PhonemeToken(
                text=move.replacement,
                kind=classify_ipa_token(move.replacement),
                editable=True,
            )
            self._set_status(f"Replaced {old.text!r} with {move.replacement!r}: {move.label}.", kind="ok")

        elif move.action == "delete":
            removed: PhonemeToken = self.tokens.pop(self.selected_index)
            self.selected_index = min(self.selected_index, max(len(self.tokens) - 1, 0))
            if self.tokens and not self.tokens[self.selected_index].editable:
                self.selected_index = self._first_editable_index(default=self.selected_index)
            self._set_status(f"Deleted {removed.text!r}.", kind="ok")

        elif move.action == "move_stress_left":
            self._move_stress(step=-1)

        elif move.action == "move_stress_right":
            self._move_stress(step=1)

        elif move.action == "insert_stress_before":
            self._insert_token_before("ˈ")

        elif move.action == "insert_copy_before":
            selected: PhonemeToken = self.tokens[self.selected_index]
            self._insert_token_before(selected.text)

        self._render()

        if self.autoplay_toggle.value:
            self.speak_current_phonemes()

    def _insert_token_before(self, text: str) -> None:
        """Insert a new editable token before the current selection."""

        token = PhonemeToken(text=text, kind=classify_ipa_token(text), editable=True)
        self.tokens.insert(self.selected_index, token)
        self._set_status(f"Inserted {text!r} before the selected phoneme.", kind="ok")

    def insert_stress_before(self) -> None:
        """Insert a primary stress marker before the selected phoneme."""

        if not self.tokens:
            return
        self.history.append(list(self.tokens))
        self._insert_token_before("ˈ")
        self._render()
        if self.autoplay_toggle.value:
            self.speak_current_phonemes()

    def insert_copy_before(self) -> None:
        """Add a new phoneme before the selection by copying the current one."""

        if not self.tokens:
            return
        self.history.append(list(self.tokens))
        selected: PhonemeToken = self.tokens[self.selected_index]
        self._insert_token_before(selected.text)
        self._render()
        if self.autoplay_toggle.value:
            self.speak_current_phonemes()

    def delete_selected(self) -> None:
        """Delete the selected token."""

        if not self.tokens:
            return
        self.apply_move(CandidateMove("down", "", "delete selected phoneme", action="delete"))

    def _on_insert_stress_clicked(self, _button: widgets.Button) -> None:
        """Handle the insert-stress action button."""

        self.insert_stress_before()

    def _on_delete_selected_clicked(self, _button: widgets.Button) -> None:
        """Handle the delete-selected action button."""

        self.delete_selected()

    def _on_insert_copy_before_clicked(self, _button: widgets.Button) -> None:
        """Handle the add-before action button."""

        self.insert_copy_before()

    def _move_stress(self, step: int) -> None:
        """Move a stress mark across neighbouring editable tokens.

        Args:
            step: Direction of movement. Use -1 for left and +1 for right.
        """

        stress_token: PhonemeToken = self.tokens.pop(self.selected_index)
        target: int = self.selected_index + step
        while 0 <= target < len(self.tokens) and not self.tokens[target].editable:
            target += step

        if step < 0:
            insert_index: int = max(0, target)
        else:
            insert_index = min(len(self.tokens), target + 1)

        self.tokens.insert(insert_index, stress_token)
        self.selected_index = insert_index
        direction_text: str = "left" if step < 0 else "right"
        self._set_status(f"Moved stress {direction_text}.", kind="ok")

    def undo(self) -> None:
        """Undo the last phoneme edit."""

        if not self.history:
            self._set_status("Nothing to undo.", kind="warn")
            return

        self.tokens = self.history.pop()
        self.selected_index = min(self.selected_index, max(len(self.tokens) - 1, 0))
        self._set_status("Undid last edit.", kind="ok")
        self._render()
        if self.autoplay_toggle.value:
            self.speak_current_phonemes()

    def speak_current_phonemes(self) -> None:
        """Generate and display audio for the current phoneme string."""

        phonemes: str = detokenise_ipa(self.tokens)
        if not phonemes:
            self._set_status("No phonemes to speak.", kind="warn")
            return

        self._set_status("Calling Kokoro /dev/generate_from_phonemes ...", kind="neutral")
        try:
            audio_bytes: bytes = self.client().generate_from_phonemes(
                phonemes=phonemes,
                voice=self.voice_input.value.strip(),
            )
        except Exception as exc:
            self._set_status(f"Speech generation failed: {type(exc).__name__}: {exc}", kind="error")
            return

        with self.audio_out:
            self.audio_out.clear_output(wait=True)
            display(Audio(data=audio_bytes, rate=24000, autoplay=True))
            display(HTML(
                "<script>"
                "setTimeout(() => {"
                "  const audioNodes = Array.from(document.querySelectorAll('audio'));"
                "  const audioNode = audioNodes[audioNodes.length - 1];"
                "  if (!audioNode) { return; }"
                "  audioNode.autoplay = true;"
                "  const playPromise = audioNode.play();"
                "  if (playPromise && typeof playPromise.catch === 'function') {"
                "    playPromise.catch(() => undefined);"
                "  }"
                "}, 0);"
                "</script>"
            ))

        self._set_status(f"Generated audio from phonemes: {phonemes}", kind="ok")

    def _on_key_event(self, event: dict[str, Any]) -> None:
        """Handle keyboard events from ipyevents.

        Args:
            event: ipyevents DOM event payload.
        """

        key: str = str(event.get("key", ""))

        if key == " ":
            self.speak_current_phonemes()
        elif key.lower() == "z":
            self.undo()
        elif key.lower() == "q":
            self.move_selection(step=-1)
        elif key.lower() == "e":
            self.move_selection(step=1)
        elif key.lower() == "w":
            self.apply_move_by_direction("up")
        elif key.lower() == "s":
            self.apply_move_by_direction("down")
        elif key.lower() == "a":
            self.apply_move_by_direction("left")
        elif key.lower() == "d":
            self.apply_move_by_direction("right")
        elif key.lower() == "r":
            self.insert_stress_before()
        elif key.lower() == "t":
            self.delete_selected()
        elif key.lower() == "f":
            self.insert_copy_before()


app = PhonemeNavigatorApp()
app.display()


/tmp/ipykernel_3865865/586549531.py:128: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.word_input.on_submit(self._on_word_submitted)
